# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
"The first finding is What Predicts Growth? It is Logistic Regression having 71% holdout accuracy of the label ML Appendix. The paper doesn't state whether the 80/20 holdout split was random by row or grouped by brand. With 57 brands contributing pages, if pages from the same brand can land on both sides of the split, the model could be partly learning brand specific writing style or template rather than a generalizable growth signal"
"The second finding is What Predicts Health? It uses Random Forest feature importance of labels Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%. The paper states Health Score is literally defined as Impressions + Position + CTR + Scroll Depth and the top four Random Forest importances are exactly those same four components, in roughly the order their point-weights would predict. A cleaner version might report residual variance the RF explains beyond a simple weighted-sum reconstruction of the known formula"

'The second finding is What Predicts Health? It uses Random Forest feature importance of labels Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%. The paper states Health Score is literally defined as Impressions + Position + CTR + Scroll Depth and the top four Random Forest importances are exactly those same four components, in roughly the order their point-weights would predict. A cleaner version might report residual variance the RF explains beyond a simple weighted-sum reconstruction of the known formula'

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
%pip -q install duckdb huggingface_hub
import duckdb
import pandas as pd
from google.colab import userdata
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DEV_MONTH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
DIM_CLIENTS = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

labels = con.sql(f"""
    SELECT content_hash_id,
      AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks END) AS clicks_first_half,
      AVG(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks END) AS clicks_second_half
    FROM read_parquet('{DEV_MONTH_PATH}')
    GROUP BY content_hash_id
""").df()
labels['is_declining'] = (labels['clicks_second_half'] < labels['clicks_first_half']).astype(int)

model_df = con.sql(f"""
    SELECT
      f.content_hash_id, f.client_hash_id,
      AVG(f.gsc_avg_position) AS avg_position,
      SUM(f.gsc_clicks) AS clicks_total,
      SUM(f.gsc_impressions) AS impressions_total,
      MAX(f.report_date) AS last_report_date
    FROM read_parquet('{DEV_MONTH_PATH}') f
    GROUP BY f.content_hash_id, f.client_hash_id
""").df()

dim = con.sql(f"""
    SELECT content_hash_id, content_type, word_count, main_intent,
           content_created_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()

model_df = model_df.merge(dim, on='content_hash_id').merge(
    labels[['content_hash_id','is_declining']], on='content_hash_id'
)
model_df['content_age_days'] = (
    pd.to_datetime(model_df['last_report_date']) - pd.to_datetime(model_df['content_created_date'])
).dt.days
model_df = model_df.dropna(subset=['avg_position','word_count','content_age_days','client_hash_id'])

num_features = ['avg_position','word_count','content_age_days','impressions_total']
cat_features = ['content_type','main_intent']

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preproc = ColumnTransformer([
    ('num', 'passthrough', num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# BEFORE: naive random row split (ignores that pages share clients)
X = model_df[num_features + cat_features]
y = model_df['is_declining']

X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(X, y, test_size=0.3, random_state=0)
rf_naive = Pipeline([('prep', preproc), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=0))])
rf_naive.fit(X_tr_naive, y_tr_naive)
naive_auc = roc_auc_score(y_te_naive, rf_naive.predict_proba(X_te_naive)[:,1])

# AFTER: grouped split by client (what I used from Week 5 onward)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]
X_tr_grp, y_tr_grp = train_df[num_features + cat_features], train_df['is_declining']
X_te_grp, y_te_grp = test_df[num_features + cat_features], test_df['is_declining']

rf_grouped = Pipeline([('prep', preproc), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=0))])
rf_grouped.fit(X_tr_grp, y_tr_grp)
grouped_auc = roc_auc_score(y_te_grp, rf_grouped.predict_proba(X_te_grp)[:,1])

print(f"Naive random split AUC:   {naive_auc:.3f}")
print(f"Grouped-by-client AUC:    {grouped_auc:.3f}")
print(f"Gap (optimism from leakage): {naive_auc - grouped_auc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Naive random split AUC:   0.838
Grouped-by-client AUC:    0.847
Gap (optimism from leakage): -0.009


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
"The impressions_total is summed across the entire March window, the same window used to compute is_declining. This means impressions_total partially overlaps the outcome period rather than being cleanly. This is a softer leak than Week 3's deliberate trap, but it's not fully clean either"
"The avg_position, word_count, content_age_days remain clean position and age are structural/observational, not aggregated over the outcome window, and word count is fixed at write time."

'The avg_position, word_count, content_age_days remain clean position and age are structural/observational, not aggregated over the outcome window, and word count is fixed at write time.'

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [10]:
"I saw in week 5 that random forest leans on impressions to predict decline. But in this observed slice, impressions showed the strongest measured importance for the decline classifier. This is directional, not causal, and may partly reflect overlap with the outcome window"
"I also wrote that position doesn't matter for this model. The position actually showed near zero permutation importance in this run which is decision support information for feature selection, not proof position is irrelevant to decline generally"

"I also wrote that position doesn't matter for this model. The position actually showed near zero permutation importance in this run which is decision support information for feature selection, not proof position is irrelevant to decline generally"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.